# Frontier Model Benchmark on math-imp-bench8

Tests frontier models via **AWS Bedrock** on `thulthula/math-imp-bench8`.

**Design:**
- `ThreadPoolExecutor` — all problems for a model fire concurrently
- **Saves a CSV after every model completes** (no lost work)
- JSONL checkpoint for within-model resume
- Retry with exponential backoff
- Supports both Converse API and invoke_model (for DeepSeek R1)

In [4]:
import os, re, json, time
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import boto3
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm  # plain tqdm, always works

/home/shivank_g/projects/atul/math-imp/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [5]:
# ── AWS Credentials ──────────────────────────────────────────────────────────
AWS_REGION     = "us-east-1"
AWS_ACCESS_KEY = "YOUR_AWS_ACCESS_KEY_HERE"
AWS_SECRET_KEY = "YOUR_AWS_SECRET_KEY_HERE"

# ── Dataset ──────────────────────────────────────────────────────────────────
DATASET_NAME = "thulthula/math-imp-bench8"

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("frontier_bench_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.jsonl"

# ── Model registry ───────────────────────────────────────────────────────────
# short_name -> (bedrock_model_id, api_type)
# api_type: "converse" (Bedrock Converse API) or "invoke" (raw invoke_model)
MODELS = {
    "sonnet-4.5":     ("us.anthropic.claude-sonnet-4-5-20250929-v1:0", "converse"),
    "deepseek-r1":    ("us.deepseek.r1-v1:0",                        "invoke"),
    "deepseek-v3.2":  ("deepseek.v3.2",                              "converse"),
    "kimi-k2.5":      ("moonshotai.kimi-k2.5",                       "converse"),
    "glm-4.7":        ("zai.glm-4.7",                                "converse"),
    "llama4-mav":     ("us.meta.llama4-maverick-17b-instruct-v1:0",  "converse"),
    "nova-premier":   ("us.amazon.nova-premier-v1:0",                "converse"),
    "mistral-lg3":    ("mistral.mistral-large-3-675b-instruct",      "converse"),
}

# ── Which models to run ──────────────────────────────────────────────────────
MODELS_TO_RUN = list(MODELS.keys())  # all, or e.g. ["sonnet-4.5", "deepseek-r1"]

# ── Concurrency ──────────────────────────────────────────────────────────────
CONCURRENCY = 10      # parallel threads per model

# ── Filters (set to None for all) ────────────────────────────────────────────
FILTER_SPLITS   = None   # e.g. ["AIME24"] or None
FILTER_VARIANTS = None   # e.g. ["original"] or None

# ── Resume ───────────────────────────────────────────────────────────────────
RESUME = True

print(f"Models:      {MODELS_TO_RUN}")
print(f"Concurrency: {CONCURRENCY} threads per model")
print(f"Resume:      {RESUME}")

Models:      ['sonnet-4.5', 'deepseek-r1', 'deepseek-v3.2', 'kimi-k2.5', 'glm-4.7', 'llama4-mav', 'nova-premier', 'mistral-lg3']
Concurrency: 10 threads per model
Resume:      True


## 2. Prompt & Answer Extraction

In [6]:
SYSTEM_PROMPT = (
    "You are an expert mathematics problem solver. "
    "Solve the given AIME problem step by step and provide your final answer "
    "in \\boxed{answer} format. Be precise — use exact values "
    "(fractions, radicals), not decimals."
)


def make_user_prompt(problem_text: str) -> str:
    return (
        "Solve this mathematics problem. Show your reasoning, "
        "then give your final answer as \\boxed{answer}.\n\n"
        f"PROBLEM:\n{problem_text}"
    )


def extract_boxed(text: str) -> list[str]:
    results, search_start = [], 0
    while True:
        idx = text.find("\\boxed{", search_start)
        if idx == -1:
            break
        start = idx + len("\\boxed{")
        depth, pos = 1, start
        while pos < len(text) and depth > 0:
            if text[pos] == "{":   depth += 1
            elif text[pos] == "}": depth -= 1
            pos += 1
        if depth == 0:
            results.append(text[start:pos - 1])
        search_start = pos
    return results


def extract_answer(response: str):
    matches = extract_boxed(response)
    if matches:
        return matches[-1].strip()
    for pattern in [
        r"(?:final\s+)?answer\s*(?:is|=|:)\s*([^\n\.]+)",
        r"therefore[,\s]+(?:the\s+answer\s+is\s+)?([^\n\.]+)",
        r"=\s*([^\n]+?)\s*$",
    ]:
        m = re.search(pattern, response, re.IGNORECASE | re.MULTILINE)
        if m:
            return m.group(1).strip().rstrip(".,;")
    return None


def normalize_answer(answer):
    if answer is None:
        return None
    s = str(answer).strip()
    s = s.replace("\\", "").replace("{", "").replace("}", "").replace("$", "")
    return " ".join(s.split()).lower()


def check_answer(predicted, correct) -> bool:
    if predicted is None or correct is None:
        return False
    p, c = normalize_answer(predicted), normalize_answer(correct)
    if p == c:
        return True
    try:
        return abs(float(p) - float(c)) < 1e-6
    except (ValueError, TypeError):
        return False


print("Helpers loaded.")

Helpers loaded.


## 3. Load Dataset

In [7]:
def load_problems():
    ds = load_dataset(DATASET_NAME)
    problems = []
    for split_name in ds:
        split = ds[split_name]
        for idx in range(len(split)):
            row = split[idx]
            base = {"split": split_name, "idx": idx,
                    "original_answer": str(row["original_answer"])}
            if row.get("original_problem"):
                problems.append({**base, "variant": "original",
                                 "problem": row["original_problem"]})
            if row.get("variation_1_node_deletion_problem"):
                problems.append({**base, "variant": "node_deletion",
                                 "problem": row["variation_1_node_deletion_problem"]})
            if row.get("variation_2_edge_deletion_problem"):
                problems.append({**base, "variant": "edge_deletion",
                                 "problem": row["variation_2_edge_deletion_problem"]})
    return problems


all_problems = load_problems()
problems = all_problems
if FILTER_SPLITS:
    problems = [p for p in problems if p["split"] in FILTER_SPLITS]
if FILTER_VARIANTS:
    problems = [p for p in problems if p["variant"] in FILTER_VARIANTS]

print(f"Dataset:  {DATASET_NAME}")
print(f"Total:    {len(all_problems)} (all variants)")
print(f"Filtered: {len(problems)}")
print(f"Splits:   {sorted(set(p['split'] for p in problems))}")
print(f"Variants: {sorted(set(p['variant'] for p in problems))}")

Dataset:  thulthula/math-imp-bench8
Total:    180 (all variants)
Filtered: 180
Splits:   ['AIME24', 'AIME25']
Variants: ['edge_deletion', 'node_deletion', 'original']


## 4. Bedrock API Caller + Checkpoint

In [8]:
# ── Thread-local boto3 clients (one per thread for thread safety) ─────────
_thread_local = threading.local()


def get_bedrock_client():
    """Get or create a thread-local bedrock-runtime client."""
    if not hasattr(_thread_local, "client"):
        _thread_local.client = boto3.client(
            "bedrock-runtime",
            region_name=AWS_REGION,
            aws_access_key_id=AWS_ACCESS_KEY,
            aws_secret_access_key=AWS_SECRET_KEY,
        )
    return _thread_local.client


def make_task_id(model, split, idx, variant):
    return f"{model}|{split}|{idx}|{variant}"


def load_checkpoint() -> set:
    done = set()
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                try:
                    done.add(json.loads(line)["task_id"])
                except (json.JSONDecodeError, KeyError):
                    pass
    return done


def load_checkpoint_records() -> list:
    records = []
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return records


_ckpt_lock = threading.Lock()

def append_checkpoint(record: dict):
    with _ckpt_lock:
        with open(CHECKPOINT_FILE, "a") as f:
            f.write(json.dumps(record) + "\n")


# ── Bedrock call via Converse API ─────────────────────────────────────────
def call_bedrock_converse(model_id: str, problem_text: str,
                          max_retries: int = 3) -> dict:
    """Call Bedrock Converse API with retry."""
    client = get_bedrock_client()
    messages = [
        {"role": "user",
         "content": [{"text": make_user_prompt(problem_text)}]},
    ]
    system = [{"text": SYSTEM_PROMPT}]

    for attempt in range(max_retries + 1):
        try:
            resp = client.converse(
                modelId=model_id,
                messages=messages,
                system=system,
                inferenceConfig={"maxTokens": 4096, "temperature": 0.0},
            )
            content = resp["output"]["message"]["content"][0]["text"]
            usage = resp.get("usage", {})
            return {
                "response": content,
                "input_tokens": usage.get("inputTokens", 0),
                "output_tokens": usage.get("outputTokens", 0),
                "error": None,
            }
        except client.exceptions.ThrottlingException:
            time.sleep(min(2 ** attempt * 2, 60))
        except Exception as e:
            if attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            return {"response": None, "input_tokens": 0,
                    "output_tokens": 0, "error": str(e)[:500]}

    return {"response": None, "input_tokens": 0, "output_tokens": 0,
            "error": "max retries exhausted"}


# ── Bedrock call via invoke_model (for DeepSeek R1 etc.) ──────────────────
def call_bedrock_invoke(model_id: str, problem_text: str,
                        max_retries: int = 3) -> dict:
    """Call Bedrock invoke_model for models that don't support Converse."""
    client = get_bedrock_client()
    body = json.dumps({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_user_prompt(problem_text)},
        ],
        "max_tokens": 16384,
        "temperature": 0.0,
    })

    for attempt in range(max_retries + 1):
        try:
            resp = client.invoke_model(
                modelId=model_id, body=body,
                contentType="application/json",
                accept="application/json",
            )
            result = json.loads(resp["body"].read())
            choice = result["choices"][0]["message"]
            # DeepSeek R1: answer in "content", reasoning in "reasoning_content"
            content = choice.get("content") or ""
            reasoning = choice.get("reasoning_content") or ""
            full_text = (reasoning + "\n" + content).strip()
            usage = result.get("usage", {})
            return {
                "response": full_text if full_text else None,
                "input_tokens": usage.get("prompt_tokens", 0),
                "output_tokens": usage.get("completion_tokens", 0),
                "error": None,
            }
        except Exception as e:
            if "Throttling" in str(type(e).__name__):
                time.sleep(min(2 ** attempt * 2, 60))
                continue
            if attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            return {"response": None, "input_tokens": 0,
                    "output_tokens": 0, "error": str(e)[:500]}

    return {"response": None, "input_tokens": 0, "output_tokens": 0,
            "error": "max retries exhausted"}


print("Bedrock callers loaded.")

Bedrock callers loaded.


## 5. Per-Model Runner (saves CSV after each)

In [9]:
def process_one(model_name, model_id, api_type, problem):
    """Process a single (model, problem) pair. Runs in a thread."""
    task_id = make_task_id(model_name, problem["split"],
                           problem["idx"], problem["variant"])

    if api_type == "invoke":
        result = call_bedrock_invoke(model_id, problem["problem"])
    else:
        result = call_bedrock_converse(model_id, problem["problem"])

    predicted_answer = None
    is_correct = False
    if result["response"]:
        predicted_answer = extract_answer(result["response"])
        if problem["variant"] == "original":
            is_correct = check_answer(predicted_answer,
                                       problem["original_answer"])

    record = {
        "task_id": task_id,
        "model": model_name,
        "model_id": model_id,
        "split": problem["split"],
        "problem_idx": problem["idx"],
        "variant": problem["variant"],
        "problem_text": problem["problem"][:2000],
        "original_answer": problem["original_answer"],
        "predicted_answer": predicted_answer,
        "is_correct_exact": is_correct,
        "full_response": (result["response"][:8000]
                          if result["response"] else None),
        "input_tokens": result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "error": result["error"],
        "timestamp": datetime.utcnow().isoformat(),
    }
    append_checkpoint(record)
    return record


def save_model_csv(model_name, records):
    df = pd.DataFrame(records)
    csv_path = OUTPUT_DIR / f"results_{model_name}.csv"
    df.to_csv(csv_path, index=False)
    return csv_path


def print_model_summary(model_name, records):
    df = pd.DataFrame(records)
    total = len(df)
    errors = int(df["error"].notna().sum())
    in_tok = int(df["input_tokens"].sum())
    out_tok = int(df["output_tokens"].sum())
    print(f"\n  {model_name}: {total} problems, "
          f"{in_tok:,} in / {out_tok:,} out tokens, {errors} errors")
    for variant in sorted(df["variant"].unique()):
        vdf = df[df["variant"] == variant]
        correct = int(vdf["is_correct_exact"].sum())
        vtotal = len(vdf)
        acc = correct / vtotal * 100 if vtotal else 0
        print(f"    {variant:<18} {correct}/{vtotal} = {acc:.1f}%")


def run_single_model(model_name, model_id, api_type, problems,
                     concurrency, done_ids):
    """Run all problems for ONE model via ThreadPoolExecutor."""
    tasks = [p for p in problems
             if make_task_id(model_name, p["split"], p["idx"],
                             p["variant"]) not in done_ids]
    if not tasks:
        print(f"  {model_name}: all {len(problems)} tasks already done (skipped)")
        return []

    print(f"  {model_name} ({model_id}): {len(tasks)} calls, "
          f"{concurrency} threads")

    records = []
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = {
            pool.submit(process_one, model_name, model_id, api_type, p): p
            for p in tasks
        }
        pbar = tqdm(total=len(futures), desc=model_name, unit="call")
        for future in as_completed(futures):
            try:
                records.append(future.result())
            except Exception as e:
                print(f"    Exception: {e}")
            pbar.update(1)
        pbar.close()

    return records


print("Runner loaded.")

Runner loaded.


## 6. Run Benchmark (model by model, CSV after each)

In [7]:
# ── Clear checkpoint if not resuming ─────────────────────────────────────────
if not RESUME and CHECKPOINT_FILE.exists():
    CHECKPOINT_FILE.unlink()
    print("Cleared previous checkpoint.")

done_ids = load_checkpoint() if RESUME else set()
if done_ids:
    print(f"Resuming: {len(done_ids)} tasks already completed")

print(f"\n{'='*70}")
print(f"FRONTIER MODEL BENCHMARK (AWS Bedrock)")
print(f"Dataset:     {DATASET_NAME}")
print(f"Models:      {MODELS_TO_RUN}")
print(f"Problems:    {len(problems)} per model")
print(f"Total calls: {len(problems) * len(MODELS_TO_RUN)} (minus done)")
print(f"{'='*70}")

all_model_records = {}
saved_csvs = {}
bench_start = time.time()

Resuming: 848 tasks already completed

FRONTIER MODEL BENCHMARK (AWS Bedrock)
Dataset:     thulthula/math-imp-bench8
Models:      ['sonnet-4.5', 'deepseek-r1', 'deepseek-v3.2', 'kimi-k2.5', 'glm-4.7', 'llama4-mav', 'nova-premier', 'mistral-lg3']
Problems:    180 per model
Total calls: 1440 (minus done)


In [8]:
for model_name in MODELS_TO_RUN:
    model_id, api_type = MODELS[model_name]
    t0 = time.time()

    print(f"\n{'─'*70}")
    print(f"MODEL: {model_name}  (api: {api_type})")
    print(f"{'─'*70}")

    # ── Run all problems concurrently ─────────────────────────────────────
    records = run_single_model(
        model_name, model_id, api_type,
        problems, CONCURRENCY, done_ids,
    )
    elapsed = time.time() - t0

    # ── Collect all records for this model (including previous runs) ──────
    all_ckpt = load_checkpoint_records()
    model_ckpt = [r for r in all_ckpt if r["model"] == model_name]
    deduped = {r["task_id"]: r for r in model_ckpt}
    final_records = list(deduped.values())
    all_model_records[model_name] = final_records

    # ── Save per-model CSV ────────────────────────────────────────────────
    if final_records:
        csv_path = save_model_csv(model_name, final_records)
        saved_csvs[model_name] = csv_path
        print(f"\n  >>> Saved: {csv_path} ({len(final_records)} rows)")
        print_model_summary(model_name, final_records)
    else:
        print(f"  No records for {model_name}.")

    print(f"  Time: {elapsed:.1f}s")
    done_ids = load_checkpoint()

total_elapsed = time.time() - bench_start
print(f"\n{'='*70}")
print(f"All models done in {total_elapsed:.1f}s")
print(f"Per-model CSVs: {list(saved_csvs.keys())}")
print(f"{'='*70}")


──────────────────────────────────────────────────────────────────────
MODEL: sonnet-4.5  (api: converse)
──────────────────────────────────────────────────────────────────────
  sonnet-4.5: all 180 tasks already done (skipped)

  >>> Saved: frontier_bench_results/results_sonnet-4.5.csv (180 rows)

  sonnet-4.5: 180 problems, 35,367 in / 220,789 out tokens, 0 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           27/60 = 45.0%
  Time: 0.0s

──────────────────────────────────────────────────────────────────────
MODEL: deepseek-r1  (api: invoke)
──────────────────────────────────────────────────────────────────────
  deepseek-r1: all 180 tasks already done (skipped)

  >>> Saved: frontier_bench_results/results_deepseek-r1.csv (180 rows)

  deepseek-r1: 180 problems, 0 in / 0 out tokens, 36 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           48/60 = 80.0%
  Time: 0.0s

───────────────────────────

kimi-k2.5:   0%|          | 0/111 [00:00<?, ?call/s]/tmp/ipykernel_2678248/242946773.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
kimi-k2.5: 100%|██████████| 111/111 [09:10<00:00,  4.96s/call]



  >>> Saved: frontier_bench_results/results_kimi-k2.5.csv (180 rows)

  kimi-k2.5: 180 problems, 32,529 in / 435,612 out tokens, 0 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           38/60 = 63.3%
  Time: 550.7s

──────────────────────────────────────────────────────────────────────
MODEL: glm-4.7  (api: converse)
──────────────────────────────────────────────────────────────────────
  glm-4.7 (zai.glm-4.7): 180 calls, 10 threads


glm-4.7:   0%|          | 0/180 [00:00<?, ?call/s]/tmp/ipykernel_2678248/242946773.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
glm-4.7: 100%|██████████| 180/180 [09:28<00:00,  3.16s/call]



  >>> Saved: frontier_bench_results/results_glm-4.7.csv (180 rows)

  glm-4.7: 180 problems, 32,331 in / 537,964 out tokens, 0 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           29/60 = 48.3%
  Time: 568.6s

──────────────────────────────────────────────────────────────────────
MODEL: llama4-mav  (api: converse)
──────────────────────────────────────────────────────────────────────
  llama4-mav (us.meta.llama4-maverick-17b-instruct-v1:0): 180 calls, 10 threads


...


  >>> Saved: frontier_bench_results/results_llama4-mav.csv (180 rows)

  llama4-mav: 180 problems, 36,383 in / 232,087 out tokens, 0 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           15/60 = 25.0%
  Time: 116.8s

──────────────────────────────────────────────────────────────────────
MODEL: nova-premier  (api: converse)
──────────────────────────────────────────────────────────────────────
  nova-premier (us.amazon.nova-premier-v1:0): 180 calls, 10 threads


nova-premier:   0%|          | 0/180 [00:00<?, ?call/s]/tmp/ipykernel_2678248/242946773.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
nova-premier: 100%|██████████| 180/180 [45:10<00:00, 15.06s/call] 



  >>> Saved: frontier_bench_results/results_nova-premier.csv (180 rows)

  nova-premier: 180 problems, 38,648 in / 169,484 out tokens, 1 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           10/60 = 16.7%
  Time: 2710.9s

──────────────────────────────────────────────────────────────────────
MODEL: mistral-lg3  (api: converse)
──────────────────────────────────────────────────────────────────────
  mistral-lg3 (mistral.mistral-large-3-675b-instruct): 180 calls, 10 threads


mistral-lg3:   0%|          | 0/180 [00:00<?, ?call/s]/tmp/ipykernel_2678248/242946773.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
mistral-lg3: 100%|██████████| 180/180 [06:37<00:00,  2.21s/call]



  >>> Saved: frontier_bench_results/results_mistral-lg3.csv (180 rows)

  mistral-lg3: 180 problems, 32,546 in / 639,937 out tokens, 0 errors
    edge_deletion      0/60 = 0.0%
    node_deletion      0/60 = 0.0%
    original           23/60 = 38.3%
  Time: 397.5s

All models done in 4346.9s
Per-model CSVs: ['sonnet-4.5', 'deepseek-r1', 'deepseek-v3.2', 'kimi-k2.5', 'glm-4.7', 'llama4-mav', 'nova-premier', 'mistral-lg3']


## 7. Aggregate All Results

In [9]:
all_ckpt = load_checkpoint_records()
deduped = {r["task_id"]: r for r in all_ckpt}
df = pd.DataFrame(list(deduped.values()))

print(f"Total records (deduped): {len(df)}")
print(f"Models: {sorted(df['model'].unique())}")

# Save combined CSVs
full_csv = OUTPUT_DIR / "all_models_full.csv"
df.to_csv(full_csv, index=False)
print(f"Saved: {full_csv}")

compact_cols = [
    "model", "split", "problem_idx", "variant",
    "original_answer", "predicted_answer", "is_correct_exact",
    "input_tokens", "output_tokens", "error",
]
compact_df = df[[c for c in compact_cols if c in df.columns]]
compact_csv = OUTPUT_DIR / "all_models_compact.csv"
compact_df.to_csv(compact_csv, index=False)
print(f"Saved: {compact_csv}")

Total records (deduped): 1679
Models: ['deepseek', 'deepseek-r1', 'deepseek-v3.2', 'glm-4.7', 'grok', 'kimi-k2.5', 'llama4-mav', 'mistral-lg3', 'nova-premier', 'opus-4.6', 'sonnet-4.5']
Saved: frontier_bench_results/all_models_full.csv
Saved: frontier_bench_results/all_models_compact.csv


## 8. Results Summary

In [10]:
total_in = int(df["input_tokens"].sum())
total_out = int(df["output_tokens"].sum())
total_errors = int(df["error"].notna().sum())

print(f"Tokens: {total_in:,} in + {total_out:,} out = {total_in+total_out:,}")
print(f"Errors: {total_errors}/{len(df)}")

summary_rows = []
print(f"\n{'Model':<18} {'Split':<10} {'Variant':<18} "
      f"{'Correct':<10} {'Total':<8} {'Acc%':<8} {'Errors':<8}")
print("-" * 85)

for model in sorted(df["model"].unique()):
    mdf = df[df["model"] == model]
    for split in sorted(mdf["split"].unique()):
        sdf = mdf[mdf["split"] == split]
        for variant in sorted(sdf["variant"].unique()):
            vdf = sdf[sdf["variant"] == variant]
            total = len(vdf)
            correct = int(vdf["is_correct_exact"].sum())
            errs = int(vdf["error"].notna().sum())
            acc = correct / total * 100 if total else 0
            print(f"{model:<18} {split:<10} {variant:<18} "
                  f"{correct:<10} {total:<8} {acc:<8.1f} {errs:<8}")
            summary_rows.append({
                "model": model, "split": split, "variant": variant,
                "correct": correct, "total": total,
                "accuracy": round(acc, 1), "errors": errs,
            })

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nSaved: {summary_csv}")

Tokens: 298,079 in + 3,270,751 out = 3,568,830
Errors: 60/1679

Model              Split      Variant            Correct    Total    Acc%     Errors  
-------------------------------------------------------------------------------------
deepseek           AIME24     original           10         29       34.5     19      
deepseek-r1        AIME24     edge_deletion      0          30       0.0      5       
deepseek-r1        AIME24     node_deletion      0          30       0.0      6       
deepseek-r1        AIME24     original           26         30       86.7     3       
deepseek-r1        AIME25     edge_deletion      0          30       0.0      8       
deepseek-r1        AIME25     node_deletion      0          30       0.0      9       
deepseek-r1        AIME25     original           22         30       73.3     5       
deepseek-v3.2      AIME24     edge_deletion      0          30       0.0      0       
deepseek-v3.2      AIME24     node_deletion      0          30     

In [11]:
# ── Leaderboard ──────────────────────────────────────────────────────────────
print(f"{'='*70}")
print("MODEL LEADERBOARD")
print(f"{'='*70}")
print(f"\n{'Model':<18} {'Original':<12} {'NodeDel':<12} "
      f"{'EdgeDel':<12} {'Overall':<12}")
print("-" * 70)

leaderboard = []
for model in sorted(df["model"].unique()):
    mdf = df[df["model"] == model]
    parts = {}
    for v in ["original", "node_deletion", "edge_deletion"]:
        vdf = mdf[mdf["variant"] == v]
        if len(vdf) > 0:
            parts[v] = f"{vdf['is_correct_exact'].sum()/len(vdf)*100:.1f}%"
        else:
            parts[v] = "N/A"
    overall = mdf["is_correct_exact"].sum() / len(mdf) * 100 if len(mdf) else 0
    leaderboard.append((model, overall))
    print(f"{model:<18} {parts['original']:<12} "
          f"{parts['node_deletion']:<12} {parts['edge_deletion']:<12} "
          f"{overall:.1f}%")

print(f"\n{'='*70}")
print("RANKED BY OVERALL ACCURACY")
print(f"{'='*70}")
for rank, (model, acc) in enumerate(
    sorted(leaderboard, key=lambda x: -x[1]), 1
):
    print(f"  {rank}. {model:<18} {acc:.1f}%")

MODEL LEADERBOARD

Model              Original     NodeDel      EdgeDel      Overall     
----------------------------------------------------------------------
deepseek           34.5%        N/A          N/A          34.5%
deepseek-r1        80.0%        0.0%         0.0%         26.7%
deepseek-v3.2      51.7%        0.0%         0.0%         17.2%
glm-4.7            48.3%        0.0%         0.0%         16.1%
grok               86.7%        N/A          N/A          86.7%
kimi-k2.5          63.3%        0.0%         0.0%         21.1%
llama4-mav         25.0%        0.0%         0.0%         8.3%
mistral-lg3        38.3%        0.0%         0.0%         12.8%
nova-premier       16.7%        0.0%         0.0%         5.6%
opus-4.6           80.0%        0.0%         0.0%         26.7%
sonnet-4.5         45.0%        0.0%         0.0%         15.0%

RANKED BY OVERALL ACCURACY
  1. grok               86.7%
  2. deepseek           34.5%
  3. deepseek-r1        26.7%
  4. opus-4.6      

## 9. Output Files

In [12]:
print("Output files:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"  {f.name:<45} {f.stat().st_size/1024:.1f} KB")
if CHECKPOINT_FILE.exists():
    print(f"  {CHECKPOINT_FILE.name:<45} {CHECKPOINT_FILE.stat().st_size/1024:.1f} KB")

Output files:
  all_models_compact.csv                        117.3 KB
  all_models_full.csv                           8874.8 KB
  compact_results.csv                           6.8 KB
  full_results.csv                              66.8 KB
  results_deepseek-r1.csv                       1211.8 KB
  results_deepseek-v3.2.csv                     1321.9 KB
  results_glm-4.7.csv                           1208.2 KB
  results_kimi-k2.5.csv                         1078.7 KB
  results_llama4-mav.csv                        818.1 KB
  results_mistral-lg3.csv                       1440.9 KB
  results_nova-premier.csv                      534.7 KB
  results_opus-4.6.csv                          604.5 KB
  results_sonnet-4.5.csv                        590.7 KB
  summary.csv                                   2.3 KB
  checkpoint.jsonl                              9802.2 KB


## 10. Proper Summary (separates Original accuracy from Impossible variant behavior)

In [2]:
import re as _re, json as _json, pandas as pd
from pathlib import Path

# ── Config (adjust if your output dir differs) ────────────────────────────────
_OUTPUT_DIR    = Path("frontier_bench_results")
_CKPT_FILE     = _OUTPUT_DIR / "checkpoint.jsonl"

# ── Inline loader (no dependency on earlier cells) ────────────────────────────
def _load_records():
    records = []
    if _CKPT_FILE.exists():
        with open(_CKPT_FILE) as f:
            for line in f:
                try:
                    records.append(_json.loads(line))
                except _json.JSONDecodeError:
                    pass
    return records

# ── Refusal detection ─────────────────────────────────────────────────────────
REFUSAL_PATTERNS = [
    r'\bno solution\b', r'\bno answer\b', r'\bimpossible\b',
    r'\bno valid\b', r'\bcannot be determined\b', r'\bcannot be solved\b',
    r'\bdoes not exist\b', r'\bunderdetermined\b', r'\binconsistent\b',
    r'\bnot solvable\b', r'\binsufficient information\b',
    r'\bno unique\b', r'\bindeterminate\b',
]

def detected_refusal(response_text):
    if not isinstance(response_text, str) or not response_text:
        return False
    t = response_text.lower()
    return any(_re.search(p, t) for p in REFUSAL_PATTERNS)

# ── Load data ─────────────────────────────────────────────────────────────────
_ckpt_records = _load_records()
_deduped      = {r["task_id"]: r for r in _ckpt_records}
_df           = pd.DataFrame(list(_deduped.values()))

orig_df = _df[_df["variant"] == "original"].copy()
imp_df  = _df[_df["variant"].isin(["node_deletion", "edge_deletion"])].copy()

imp_df["refused"]     = imp_df["full_response"].apply(detected_refusal)
imp_df["gave_answer"] = imp_df["predicted_answer"].notna()

models = sorted(_df["model"].unique())
print(f"Loaded {len(_df)} records | Models: {models}")

# ── 1. ORIGINAL ACCURACY ─────────────────────────────────────────────────────
print()
print("=" * 72)
print("ORIGINAL PROBLEMS — ACCURACY  (pass@1, this is the real metric)")
print("=" * 72)
print(f"{'Model':<18} {'AIME24':>10} {'AIME25':>10} {'Combined':>12}  Errors")
print("-" * 60)

for model in models:
    mdf = orig_df[orig_df["model"] == model]
    if mdf.empty:
        continue
    combined_c, combined_t = 0, 0
    parts = {}
    for split in ["AIME24", "AIME25"]:
        sdf = mdf[mdf["split"] == split]
        c, t = int(sdf["is_correct_exact"].sum()), len(sdf)
        parts[split] = f"{c}/{t} ({c/t*100:.0f}%)" if t else "N/A"
        combined_c += c; combined_t += t
    errs = int(mdf["error"].notna().sum())
    acc  = combined_c / combined_t * 100 if combined_t else 0
    combined = f"{combined_c}/{combined_t} ({acc:.1f}%)"
    print(f"{model:<18} {parts['AIME24']:>10} {parts['AIME25']:>10} {combined:>12}  {errs}")

# ── 2. IMPOSSIBLE VARIANTS — HALLUCINATION + REFUSAL ─────────────────────────
print()
print("=" * 72)
print("IMPOSSIBLE VARIANTS — node_deletion + edge_deletion")
print("  Hallucination Rate = model gave a number  (BAD  — should refuse)")
print("  Refusal Rate       = model said impossible (GOOD — detected the trap)")
print("=" * 72)
print(f"{'Model':<18} {'Variant':<16} {'Total':>6} {'Gave Answer':>12} {'Halluc%':>9} {'Refused':>9} {'Refusal%':>9}")
print("-" * 85)

for model in models:
    mdf   = imp_df[imp_df["model"] == model]
    if mdf.empty:
        continue
    first = True
    for variant in ["node_deletion", "edge_deletion"]:
        vdf = mdf[mdf["variant"] == variant]
        if vdf.empty:
            continue
        total       = len(vdf)
        gave_ans    = int(vdf["gave_answer"].sum())
        refused     = int(vdf["refused"].sum())
        halluc_pct  = gave_ans / total * 100 if total else 0
        refusal_pct = refused  / total * 100 if total else 0
        label = model if first else ""
        print(f"{label:<18} {variant:<16} {total:>6} {gave_ans:>12} {halluc_pct:>8.1f}% {refused:>9} {refusal_pct:>8.1f}%")
        first = False

# ── 3. LEADERBOARD — ranked by original accuracy ──────────────────────────────
print()
print("=" * 72)
print("LEADERBOARD  (ranked by original accuracy = pass@1)")
print("=" * 72)
print(f"{'Rank':<6} {'Model':<18} {'Orig Acc':>10} {'Halluc Rate':>13}  Note")
print("-" * 68)

board = []
for model in models:
    o = orig_df[orig_df["model"] == model]
    i = imp_df[imp_df["model"] == model]
    orig_acc   = o["is_correct_exact"].mean() * 100 if len(o) else float("nan")
    halluc_pct = i["gave_answer"].mean()       * 100 if len(i) else float("nan")
    board.append((model, orig_acc, halluc_pct))

for rank, (model, orig_acc, halluc_pct) in enumerate(
    sorted(board, key=lambda x: -x[1]), 1
):
    note = ""
    if not pd.isna(halluc_pct):
        if halluc_pct > 90:
            note = "<-- always hallucinates on impossible"
        elif halluc_pct < 20:
            note = "<-- good: mostly refuses impossible"
    imp_str = f"{halluc_pct:.1f}%" if not pd.isna(halluc_pct) else "N/A"
    print(f"{rank:<6} {model:<18} {orig_acc:>9.1f}% {imp_str:>12}  {note}")

print()
print("NOTE: Halluc Rate = % of impossible problems where model gave any number.")
print("      Lower is better — means model detected the problem was unsolvable.")


Loaded 1679 records | Models: ['deepseek', 'deepseek-r1', 'deepseek-v3.2', 'glm-4.7', 'grok', 'kimi-k2.5', 'llama4-mav', 'mistral-lg3', 'nova-premier', 'opus-4.6', 'sonnet-4.5']

ORIGINAL PROBLEMS — ACCURACY  (pass@1, this is the real metric)
Model                  AIME24     AIME25     Combined  Errors
------------------------------------------------------------
deepseek           10/29 (34%)        N/A 10/29 (34.5%)  19
deepseek-r1        26/30 (87%) 22/30 (73%) 48/60 (80.0%)  8
deepseek-v3.2      18/30 (60%) 13/30 (43%) 31/60 (51.7%)  0
glm-4.7            15/30 (50%) 14/30 (47%) 29/60 (48.3%)  0
grok               26/30 (87%)        N/A 26/30 (86.7%)  4
kimi-k2.5          22/30 (73%) 16/30 (53%) 38/60 (63.3%)  0
llama4-mav         11/30 (37%) 4/30 (13%) 15/60 (25.0%)  0
mistral-lg3        15/30 (50%) 8/30 (27%) 23/60 (38.3%)  0
nova-premier       5/30 (17%) 5/30 (17%) 10/60 (16.7%)  0
opus-4.6           30/30 (100%) 18/30 (60%) 48/60 (80.0%)  0
sonnet-4.5         16/30 (53%) 11/30 (

## 11. Pass@k Evaluation (10 samples per problem, temperature=0.7)

In [14]:
# ── Pass@k configuration ─────────────────────────────────────────────────────
N_SAMPLES    = 10
PASSATK_TEMP = 0.7
PASSATK_DIR  = OUTPUT_DIR / "passatk"
PASSATK_DIR.mkdir(exist_ok=True)
PASSATK_CKPT = PASSATK_DIR / "passatk_checkpoint.jsonl"

# Only original problems have ground truth — impossible variants can't be scored
orig_problems = [p for p in problems if p["variant"] == "original"]
print(f"Problems for pass@k : {len(orig_problems)}")
print(f"Samples per problem : {N_SAMPLES}")
print(f"Temperature         : {PASSATK_TEMP}")
print(f"Total calls/model   : {len(orig_problems) * N_SAMPLES}")
print(f"Models              : {MODELS_TO_RUN}")
print(f"Grand total calls   : {len(orig_problems) * N_SAMPLES * len(MODELS_TO_RUN)}")

# ── Temperature-aware API callers ─────────────────────────────────────────────
def call_converse_temp(model_id, problem_text, temperature=0.7, max_retries=3):
    client = get_bedrock_client()
    messages = [{"role": "user", "content": [{"text": make_user_prompt(problem_text)}]}]
    system   = [{"text": SYSTEM_PROMPT}]
    for attempt in range(max_retries + 1):
        try:
            resp    = client.converse(
                modelId=model_id, messages=messages, system=system,
                inferenceConfig={"maxTokens": 4096, "temperature": temperature},
            )
            content = resp["output"]["message"]["content"][0]["text"]
            usage   = resp.get("usage", {})
            return {"response": content,
                    "input_tokens":  usage.get("inputTokens", 0),
                    "output_tokens": usage.get("outputTokens", 0),
                    "error": None}
        except Exception as e:
            if attempt < max_retries:
                time.sleep(2 ** attempt)
            else:
                return {"response": None, "input_tokens": 0,
                        "output_tokens": 0, "error": str(e)[:300]}
    return {"response": None, "input_tokens": 0, "output_tokens": 0, "error": "max retries"}


def call_invoke_temp(model_id, problem_text, temperature=0.7, max_retries=3):
    client = get_bedrock_client()
    body = json.dumps({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": make_user_prompt(problem_text)},
        ],
        "max_tokens": 4096,
        "temperature": temperature,
    })
    for attempt in range(max_retries + 1):
        try:
            resp    = client.invoke_model(modelId=model_id, body=body,
                                          contentType="application/json",
                                          accept="application/json")
            result  = json.loads(resp["body"].read())
            choice  = result["choices"][0]["message"]
            content   = choice.get("content") or ""
            reasoning = choice.get("reasoning_content") or ""
            full_text = (reasoning + "\n" + content).strip()
            usage = result.get("usage", {})
            return {"response": full_text or None,
                    "input_tokens":  usage.get("prompt_tokens", 0),
                    "output_tokens": usage.get("completion_tokens", 0),
                    "error": None}
        except Exception as e:
            if attempt < max_retries:
                time.sleep(2 ** attempt)
            else:
                return {"response": None, "input_tokens": 0,
                        "output_tokens": 0, "error": str(e)[:300]}
    return {"response": None, "input_tokens": 0, "output_tokens": 0, "error": "max retries"}


# ── Checkpoint helpers ────────────────────────────────────────────────────────
_pk_lock = threading.Lock()

def append_passatk_ckpt(record):
    with _pk_lock:
        with open(PASSATK_CKPT, "a") as f:
            f.write(json.dumps(record) + "\n")

def load_passatk_done():
    done = set()
    if PASSATK_CKPT.exists():
        with open(PASSATK_CKPT) as f:
            for line in f:
                try:
                    r = json.loads(line)
                    done.add((r["model"], r["split"], r["problem_idx"], r["sample_idx"]))
                except Exception:
                    pass
    return done

def load_passatk_records():
    records = []
    if PASSATK_CKPT.exists():
        with open(PASSATK_CKPT) as f:
            for line in f:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records


# ── Single-sample worker ──────────────────────────────────────────────────────
def run_one_passatk(model_name, model_id, api_type, problem, sample_idx):
    if api_type == "invoke":
        result = call_invoke_temp(model_id, problem["problem"], PASSATK_TEMP)
    else:
        result = call_converse_temp(model_id, problem["problem"], PASSATK_TEMP)

    predicted  = None
    is_correct = False
    if result["response"]:
        predicted  = extract_answer(result["response"])
        is_correct = check_answer(predicted, problem["original_answer"])

    record = {
        "model":           model_name,
        "split":           problem["split"],
        "problem_idx":     problem["idx"],
        "sample_idx":      sample_idx,
        "original_answer": problem["original_answer"],
        "predicted_answer": predicted,
        "is_correct":      is_correct,
        "input_tokens":    result["input_tokens"],
        "output_tokens":   result["output_tokens"],
        "error":           result["error"],
    }
    append_passatk_ckpt(record)
    return record

print("Pass@k helpers loaded.")

Problems for pass@k : 60
Samples per problem : 10
Temperature         : 0.7
Total calls/model   : 600
Models              : ['sonnet-4.5', 'deepseek-r1', 'deepseek-v3.2', 'kimi-k2.5', 'glm-4.7', 'llama4-mav', 'nova-premier', 'mistral-lg3']
Grand total calls   : 4800
Pass@k helpers loaded.


## 12. Cleanup — Remove Non-Bedrock / Incomplete Model Records

Removes `grok` (xAI API, only AIME24 original, not on Bedrock) and `deepseek`
(non-Bedrock API, partial run) from the checkpoint.  Keeps only the 9 fully-Bedrock
models and rebuilds the aggregate CSVs.

In [ ]:
import json, pandas as pd
from pathlib import Path

_OUTPUT_DIR  = Path("frontier_bench_results")
_CKPT_FILE   = _OUTPUT_DIR / "checkpoint.jsonl"
SKIP_MODELS  = {"grok", "deepseek"}   # incomplete / non-Bedrock runs

# ── Filter checkpoint ─────────────────────────────────────────────────────────
kept, removed_count = [], 0
with open(_CKPT_FILE) as f:
    for line in f:
        try:
            r = json.loads(line)
            if r.get("model") in SKIP_MODELS:
                removed_count += 1
            else:
                kept.append(line.rstrip("\n"))
        except json.JSONDecodeError:
            pass

with open(_CKPT_FILE, "w") as f:
    f.write("\n".join(kept) + ("\n" if kept else ""))

print(f"Removed {removed_count} records from models: {SKIP_MODELS}")
print(f"Remaining in checkpoint: {len(kept)} records")

# ── Rebuild aggregate CSVs ────────────────────────────────────────────────────
_records = [json.loads(l) for l in kept]
_deduped = {r["task_id"]: r for r in _records}
_full_df = pd.DataFrame(list(_deduped.values()))
_full_df.to_csv(_OUTPUT_DIR / "all_models_full.csv", index=False)

_compact_cols = ["model", "model_id", "split", "problem_idx", "variant",
                 "original_answer", "predicted_answer", "is_correct_exact",
                 "input_tokens", "output_tokens", "error"]
_full_df[[c for c in _compact_cols if c in _full_df.columns]].to_csv(
    _OUTPUT_DIR / "all_models_compact.csv", index=False)

print(f"Rebuilt all_models_full.csv + all_models_compact.csv ({len(_full_df)} rows)")

# ── Quick accuracy check ──────────────────────────────────────────────────────
_orig = _full_df[_full_df["variant"] == "original"]
print()
print(f"{'Model':<18} {'AIME24':>10} {'AIME25':>10}  {'Combined':>14}  Errors")
print("-" * 65)
for m in sorted(_orig["model"].unique()):
    mdf = _orig[_orig["model"] == m]
    parts = {}
    for sp in ["AIME24", "AIME25"]:
        sdf = mdf[mdf["split"] == sp]
        c, t = int(sdf["is_correct_exact"].sum()), len(sdf)
        parts[sp] = f"{c}/{t}" if t else "N/A"
    c_all, t_all = int(mdf["is_correct_exact"].sum()), len(mdf)
    errs = int(mdf["error"].notna().sum())
    print(f"{m:<18} {parts['AIME24']:>10} {parts['AIME25']:>10}  "
          f"{c_all}/{t_all} ({c_all/t_all*100:.1f}%):>14  {errs}")


## 13. Rerun Errored Tasks (deepseek-r1 timeouts)

deepseek-r1 had 36 read-timeout errors.  This cell identifies the still-errored
tasks, loads the full problem text from the dataset, and retries them via Bedrock.
New results are appended to the checkpoint (last record per task_id wins on dedup).

In [ ]:
import json, time, threading
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from datasets import load_dataset
from tqdm import tqdm

_OUTPUT_DIR = Path("frontier_bench_results")
_CKPT_FILE  = _OUTPUT_DIR / "checkpoint.jsonl"
_CONCURRENCY = 10

# Bedrock model registry — only models available on Bedrock
_MODEL_REGISTRY = {
    "sonnet-4.5":    ("us.anthropic.claude-sonnet-4-5-20250929-v1:0", "converse"),
    "deepseek-r1":   ("us.deepseek.r1-v1:0",                         "invoke"),
    "deepseek-v3.2": ("deepseek.v3.2",                               "converse"),
    "kimi-k2.5":     ("moonshotai.kimi-k2.5",                        "converse"),
    "glm-4.7":       ("zai.glm-4.7",                                 "converse"),
    "llama4-mav":    ("us.meta.llama4-maverick-17b-instruct-v1:0",   "converse"),
    "nova-premier":  ("us.amazon.nova-premier-v1:0",                 "converse"),
    "mistral-lg3":   ("mistral.mistral-large-3-675b-instruct",       "converse"),
    "opus-4.6":      ("us.anthropic.claude-opus-4-6-20250801-v1:0",  "converse"),
}

# ── Find still-errored tasks ──────────────────────────────────────────────────
_all_recs = []
with open(_CKPT_FILE) as f:
    for line in f:
        try: _all_recs.append(json.loads(line))
        except: pass

_deduped = {r["task_id"]: r for r in _all_recs}
_errored = {
    tid: r for tid, r in _deduped.items()
    if r.get("error") and r.get("model") in _MODEL_REGISTRY
}

if not _errored:
    print("Nothing to retry — all Bedrock model tasks completed successfully!")
else:
    print(f"Tasks to retry: {len(_errored)}")
    _by_model = {}
    for r in _errored.values():
        _by_model.setdefault(r["model"], {})
        k = f"{r['split']} / {r['variant']}"
        _by_model[r["model"]][k] = _by_model[r["model"]].get(k, 0) + 1
    for m, sv in sorted(_by_model.items()):
        print(f"  {m}: {sum(sv.values())} errors")
        for k, v in sorted(sv.items()):
            print(f"    {k}: {v}")

    # ── Load full problem texts from dataset ──────────────────────────────────
    print("\nLoading dataset...")
    _ds = load_dataset("thulthula/math-imp-bench8")
    _prob_lookup = {}   # (split, idx, variant) -> (full_text, original_answer)
    for _split_name in _ds:
        for _idx in range(len(_ds[_split_name])):
            _row = _ds[_split_name][_idx]
            _ans = str(_row["original_answer"])
            if _row.get("original_problem"):
                _prob_lookup[(_split_name, _idx, "original")] = (_row["original_problem"], _ans)
            if _row.get("variation_1_node_deletion_problem"):
                _prob_lookup[(_split_name, _idx, "node_deletion")] = (_row["variation_1_node_deletion_problem"], _ans)
            if _row.get("variation_2_edge_deletion_problem"):
                _prob_lookup[(_split_name, _idx, "edge_deletion")] = (_row["variation_2_edge_deletion_problem"], _ans)
    print(f"Loaded {len(_prob_lookup)} problems from dataset")

    # ── Retry worker ─────────────────────────────────────────────────────────
    _retry_lock = threading.Lock()

    def _append_retry(rec):
        with _retry_lock:
            with open(_CKPT_FILE, "a") as f:
                f.write(json.dumps(rec) + "\n")

    def _retry_one(task_id, old_rec):
        model_name = old_rec["model"]
        model_id, api_type = _MODEL_REGISTRY[model_name]
        split, idx, variant = old_rec["split"], old_rec["problem_idx"], old_rec["variant"]
        prob_entry = _prob_lookup.get((split, idx, variant))
        if not prob_entry:
            return None
        problem_text, orig_ans = prob_entry

        if api_type == "invoke":
            result = call_bedrock_invoke(model_id, problem_text)
        else:
            result = call_bedrock_converse(model_id, problem_text)

        predicted, is_correct = None, False
        if result["response"]:
            predicted  = extract_answer(result["response"])
            if variant == "original":
                is_correct = check_answer(predicted, orig_ans)

        rec = {
            "task_id":          task_id,
            "model":            model_name,
            "model_id":         model_id,
            "split":            split,
            "problem_idx":      idx,
            "variant":          variant,
            "problem_text":     problem_text[:2000],
            "original_answer":  orig_ans,
            "predicted_answer": predicted,
            "is_correct_exact": is_correct,
            "full_response":    result["response"][:8000] if result["response"] else None,
            "input_tokens":     result["input_tokens"],
            "output_tokens":    result["output_tokens"],
            "error":            result["error"],
            "timestamp":        datetime.utcnow().isoformat(),
        }
        _append_retry(rec)
        return rec

    # ── Run retries ───────────────────────────────────────────────────────────
    print(f"\nRetrying {len(_errored)} tasks ({_CONCURRENCY} threads)...")
    _new_recs = []
    with ThreadPoolExecutor(max_workers=_CONCURRENCY) as pool:
        futures = {pool.submit(_retry_one, tid, r): tid for tid, r in _errored.items()}
        pbar = tqdm(total=len(futures), desc="retry", unit="task")
        for fut in as_completed(futures):
            try:
                rec = fut.result()
                if rec: _new_recs.append(rec)
            except Exception as e:
                print(f"  Exception: {e}")
            pbar.update(1)
        pbar.close()

    # ── Summary ───────────────────────────────────────────────────────────────
    still_err = sum(1 for r in _new_recs if r.get("error"))
    fixed     = len(_new_recs) - still_err
    print(f"\nRetry done: {fixed} fixed, {still_err} still errored")

    # Rebuild CSVs with deduped (last record per task_id wins)
    _all2 = []
    with open(_CKPT_FILE) as f:
        for line in f:
            try: _all2.append(json.loads(line))
            except: pass
    _deduped2 = {r["task_id"]: r for r in _all2}
    import pandas as pd
    pd.DataFrame(list(_deduped2.values())).to_csv(
        _OUTPUT_DIR / "all_models_full.csv", index=False)
    print(f"Rebuilt all_models_full.csv ({len(_deduped2)} rows)")

    # Final accuracy for retried models
    _orig2 = pd.DataFrame([r for r in _deduped2.values() if r["variant"] == "original"])
    print("\nOriginal accuracy after retry:")
    for m in sorted(_orig2["model"].unique()):
        mdf = _orig2[_orig2["model"] == m]
        errs = int(mdf["error"].notna().sum())
        if errs == 0 and m not in _by_model:
            continue
        c, t = int(mdf["is_correct_exact"].sum()), len(mdf)
        print(f"  {m:<18} {c}/{t} ({c/t*100:.1f}%)  errors remaining: {errs}")
